# 02 – Explore SCB population grid

**Goal:** check that the SCB 1 km grid is complete and correct before we use it to compute distances.

**Data:** `raw.scb_grid_1km`, loaded by `hamburgarkartan.scb_grid` from SCB's WFS service (year **2025**, SWEREF 99 TM).

Run `uv run python -m hamburgarkartan.scb_grid` first.

In [3]:
import duckdb
from hamburgarkartan.scb_grid import DB_PATH

# read_only = we only look, and other programs can still use the database
con = duckdb.connect(str(DB_PATH), read_only=True)
con.sql("INSTALL spatial; LOAD spatial;")  # needed for geometry functions

ModuleNotFoundError: No module named 'hamburgarkartan.scb_grid'

## 1. Which columns did SCB give us?
We don't know SCB's column names in advance – this shows them.

In [ ]:
con.sql("DESCRIBE raw.scb_grid_1km")

**Finding:** fill in – which column is the population, which is the geometry, is there a cell id?

## 2. A few example rows

In [ ]:
con.sql("SELECT * FROM raw.scb_grid_1km LIMIT 5")

## 3. Pick the columns to use
The cell guesses the geometry and population columns. **Check the guess** and change it if it's wrong.

In [ ]:
cols = con.sql("DESCRIBE raw.scb_grid_1km").fetchall()  # (name, type, ...)

# geometry = the column with type GEOMETRY
GEOM = next(name for name, typ, *_ in cols if typ == "GEOMETRY")

# population = first number column whose name looks like population
numeric = [name for name, typ, *_ in cols if typ in ("BIGINT", "INTEGER", "DOUBLE", "HUGEINT", "SMALLINT", "FLOAT") or typ.startswith("DECIMAL")]
POP = next((n for n in numeric if any(k in n.lower() for k in ("beftot", "tot", "bef", "pop"))), None)

print("geometry column:  ", GEOM)
print("population column:", POP)
# If a guess is wrong, set it by hand, e.g.:  POP = "beftotalt"

## 4. Did we get the whole grid?
Some WFS servers stop after a fixed number of features. A **round number** (10 000, 100 000, 1 000 000 …) means the download was probably cut off.

In [ ]:
cells = con.sql("SELECT count(*) FROM raw.scb_grid_1km").fetchone()[0]
print(f"grid cells: {cells:,}".replace(",", " "))

if cells % 1000 == 0:
    print("⚠️  Round number – the server may have cut the download. Check before continuing!")
else:
    print("✅ Not a round number – looks complete.")

**Finding:** fill in – number of cells. The project plan estimated **> 200 000**.

## 5. Is the geometry what we expect?
- Type: polygon (a square per cell)
- Area: about **1 000 000 m²** (= 1 km²)
- Coordinates in SWEREF 99 TM: x ≈ 200 000–950 000, y ≈ 6 100 000–7 700 000

In [ ]:
con.sql(f"""
    SELECT
        ST_GeometryType({GEOM}) AS type,
        round(avg(ST_Area({GEOM})))  AS avg_area_m2,
        round(min(ST_Area({GEOM})))  AS min_area_m2,
        round(max(ST_Area({GEOM})))  AS max_area_m2,
        min(ST_XMin({GEOM})) AS min_x, max(ST_XMax({GEOM})) AS max_x,
        min(ST_YMin({GEOM})) AS min_y, max(ST_YMax({GEOM})) AS max_y,
        count(*) AS cells
    FROM raw.scb_grid_1km
    GROUP BY 1
""")

**Finding:** fill in – is every cell a 1 km² square, and are the coordinates in SWEREF 99 TM (meters, not degrees)?

## 6. Does the population add up?
Sweden has about **10.6 million** inhabitants. SCB adjusts small numbers for privacy, so the total can differ slightly.

In [ ]:
con.sql(f"""
    SELECT
        sum({POP})                                 AS total_population,
        count(*) FILTER (WHERE {POP} IS NULL)      AS cells_missing_pop,
        count(*) FILTER (WHERE {POP} = 0)          AS cells_with_0,
        min({POP})                                 AS min_pop,
        max({POP})                                 AS max_pop,
        round(avg({POP}), 1)                       AS avg_pop
    FROM raw.scb_grid_1km
""")

**Finding:** fill in – total population, and are empty cells (0 inhabitants) included or not?

## 7. Any duplicate cells?
Two cells at the same place would count people twice.

In [ ]:
con.sql(f"""
    SELECT count(*) AS cells, count(DISTINCT ST_AsText(ST_Centroid({GEOM}))) AS unique_positions
    FROM raw.scb_grid_1km
""")

**Finding:** `cells` and `unique_positions` should be equal.

## 8. How is the population spread?
Most of Sweden is empty – many cells have very few people.

In [ ]:
con.sql(f"""
    SELECT
        CASE
            WHEN {POP} < 10    THEN '1) 1–9'
            WHEN {POP} < 100   THEN '2) 10–99'
            WHEN {POP} < 1000  THEN '3) 100–999'
            WHEN {POP} < 10000 THEN '4) 1 000–9 999'
            ELSE                    '5) 10 000+'
        END AS people_per_cell,
        count(*) AS cells,
        sum({POP}) AS people,
        round(100 * sum({POP}) / sum(sum({POP})) OVER (), 1) AS pct_of_people
    FROM raw.scb_grid_1km
    WHERE {POP} > 0
    GROUP BY 1
    ORDER BY 1
""")

**Finding:** fill in – e.g. "X % of the cells hold only Y % of the people". This matters for the hamburger deserts: remote cells are many, but few people live there.

## 9. Quick map
Each dot is one grid cell, darker = more people. It should look like Sweden.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = con.sql(f"""
    SELECT ST_X(ST_Centroid({GEOM})) AS x, ST_Y(ST_Centroid({GEOM})) AS y, {POP} AS pop
    FROM raw.scb_grid_1km
    WHERE {POP} > 0
""").df()

fig, ax = plt.subplots(figsize=(5, 9))
ax.scatter(df.x, df.y, c=np.log10(df["pop"]), s=0.2, cmap="viridis_r")
ax.set_aspect("equal")
ax.set_title("Population per 1 km cell (log scale)")
ax.set_axis_off()
plt.show()

**Finding:** fill in – does the shape look like Sweden, with the big cities darkest?

## Summary
- Cells loaded: _fill in_ (complete / cut off?)
- Geometry: _fill in_ (1 km² squares in SWEREF 99 TM → no transform needed)
- Total population: _fill in_ vs ~10.6 million
- Empty cells included: _yes / no_

**Next:** write `stg_scb__grid_1km` in dbt: rename columns, keep cells with population > 0, add the cell centre point used for distances.

In [ ]:
con.close()